# Tournament

In [12]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [13]:
print("Compiling full 2019 2500-tick bars...")
bars_list = []

for m in range(1, 13):
    month_str = f"2019{m:02d}"
    print(f"Loading {month_str}...")
    
    # 1. Load Parquet
    bid = pd.read_parquet(f"../data/processed/eurusd_dukascopy_bid_{month_str}.parquet")
    ask = pd.read_parquet(f"../data/processed/eurusd_dukascopy_ask_{month_str}.parquet")
    
    # 2. Sort (required for merge_asof)
    bid['datetime'] = pd.to_datetime(bid['datetime'])
    ask['datetime'] = pd.to_datetime(ask['datetime'])
    bid = bid.sort_values('datetime')
    ask = ask.sort_values('datetime')
    
    # 3. Merge and Mid-Price
    merged = pd.merge_asof(
        bid, 
        ask[['datetime', 'price']], 
        on='datetime', 
        direction='backward', 
        suffixes=('_bid', '_ask')
    )
    merged['mid_price'] = (merged['price_bid'] + merged['price_ask']) / 2
    
    # 4. Sample immediately to save memory (2500 ticks)
    sampled = merged.iloc[::2500].copy()
    bars_list.append(sampled)

# 5. Combine the whole year into df_bars
df_bars = pd.concat(bars_list).reset_index(drop=True)
df_bars.set_index('datetime', inplace=True)

# 6. Calculate log returns for the whole continuous year
df_bars['log_returns'] = np.log(df_bars['mid_price'] / df_bars['mid_price'].shift(1))
df_bars.dropna(inplace=True)

print(f"\nSuccess! Total 2019 bars ready: {len(df_bars)}")

Compiling full 2019 2500-tick bars...
Loading 201901...
Loading 201902...
Loading 201903...
Loading 201904...
Loading 201905...
Loading 201906...
Loading 201907...
Loading 201908...
Loading 201909...
Loading 201910...
Loading 201911...
Loading 201912...

Success! Total 2019 bars ready: 11681


In [ ]:
# 1. Prepare the Target Variable
# Assuming df_bars is your full 2019 dataframe of 2500-tick bars
df_bars['vol_proxy'] = np.abs(df_bars['log_returns']) * 1e6
df_bars['month'] = df_bars.index.month
df_bars.dropna(inplace=True)

results = []

print("Starting Walk-Forward Volatility Tournament...")

# 2. Walk-Forward Loop (Train on 3 months, Test on the 4th)
for m in range(1, 10): # Loop from Jan to Sep (so last test is Dec)
    train_months = [m, m+1, m+2]
    test_month = m + 3
    
    # Split the data
    train_data = df_bars[df_bars['month'].isin(train_months)]['vol_proxy']
    test_data = df_bars[df_bars['month'] == test_month]['vol_proxy']
    
    if len(test_data) == 0:
        continue
        
    print(f"\nTraining on Months {train_months} | Testing on Month {test_month}")
    
    # To do 1-step ahead forecasting, we need the combined series
    full_data = pd.concat([train_data, test_data])
    start_idx = len(train_data)
    end_idx = len(full_data) - 1

    # --- MODEL 1: The Baseline (SMA of last 5 bars) ---
    # A simple moving average of the last 5 periods' volatility
    sma_forecast = full_data.rolling(window=5).mean().shift(1).iloc[start_idx:]
    sma_rmse = np.sqrt(mean_squared_error(test_data, sma_forecast.fillna(method='bfill')))

    # --- MODEL 2: AR(1) ---
    print("  Fitting AR(1)...")
    ar_model = AutoReg(train_data, lags=1).fit()
    # Predict the test month using 1-step-ahead dynamic=False forecasting
    ar_forecast = ar_model.predict(start=start_idx, end=end_idx, dynamic=False)
    ar_rmse = np.sqrt(mean_squared_error(test_data, ar_forecast))

    # --- MODEL 3: Markov-Switching AR(1) (The Filtered Hack) ---
    print("  Fitting MS-AR(1)...")
    try:
        # Feed the entire 4-month block (Train + Test)
        ms_model = sm.tsa.MarkovAutoregression(
            endog=full_data, 
            k_regimes=2, 
            order=1, 
            switching_ar=False, 
            switching_variance=True
        )
        
        # 'bfgs' is more stable, 'disp=False' keeps it quiet
        ms_res = ms_model.fit(method='bfgs', disp=False)
        
        # THE FIX: Grab strictly the last N elements to match test_data perfectly
        ms_forecast = ms_res.fittedvalues[-len(test_data):]
        ms_rmse = np.sqrt(mean_squared_error(test_data, ms_forecast))
        
    except Exception as e:
        print(f"  MS-AR(1) Failed: {repr(e)}")
        ms_rmse = np.nan
        
    except Exception as e:
        print(f"  MS-AR(1) Failed: {repr(e)}")
        ms_rmse = np.nan
        
    except Exception as e:
        print(f"  MS-AR(1) Failed: {repr(e)}") # Added repr() to print the exact math error if it fails
        ms_rmse = np.nan
        
    except Exception as e:
        print(f"  MS-AR(1) Failed: {e}")
        ms_rmse = np.nan

    # Record Results
    results.append({
        'Test Month': test_month,
        'SMA RMSE': round(sma_rmse, 2),
        'AR(1) RMSE': round(ar_rmse, 2),
        'MS-AR(1) RMSE': round(ms_rmse, 2)
    })

# 3. Print the Final Scoreboard
results_df = pd.DataFrame(results)
print("\n=== FINAL TOURNAMENT RESULTS (RMSE) ===")
print("Lower is better. RMSE represents the average error in scaled volatility units.")
print(results_df.to_string(index=False))

Starting Walk-Forward Volatility Tournament...

Training on Months [1, 2, 3] | Testing on Month 4
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [2, 3, 4] | Testing on Month 5
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [3, 4, 5] | Testing on Month 6
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [4, 5, 6] | Testing on Month 7
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [5, 6, 7] | Testing on Month 8
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [6, 7, 8] | Testing on Month 9
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [7, 8, 9] | Testing on Month 10
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [8, 9, 10] | Testing on Month 11
  Fitting AR(1)...
  Fitting MS-AR(1)...

Training on Months [9, 10, 11] | Testing on Month 12
  Fitting AR(1)...
  Fitting MS-AR(1)...
